In [ ]:
import os
import json
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, rand, lit, floor, stddev, desc, coalesce
from pyspark.sql.window import Window

os.makedirs("/workspace/outputs/ncf", exist_ok=True)

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("6_Prepare_NCF_Dataset_With_Genome")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.instances", "1")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"
HDFS_GOLD = "hdfs://namenode:8020/netflix-recsys/gold"

interactions = spark.read.parquet(f"{HDFS_GOLD}/labeled_interactions")
genome_scores = spark.read.parquet(f"{HDFS_RAW}/genome_scores")
genome_tags = spark.read.parquet(f"{HDFS_RAW}/genome_tags")

In [ ]:
positive = (
    interactions
    .filter(col("label") == 1)
    .select("userId", "movieId", "label")
)

positive_sample = positive.orderBy(rand(seed=42)).limit(2_000_000)

In [ ]:
users = (
    positive_sample
    .select("userId")
    .distinct()
    .orderBy("userId")
    .withColumn("user_idx", row_number().over(Window.orderBy("userId")) - 1)
)

movies = (
    positive_sample
    .select("movieId")
    .distinct()
    .orderBy("movieId")
    .withColumn("movie_idx", row_number().over(Window.orderBy("movieId")) - 1)
)

positive_idx = (
    positive_sample
    .join(users, "userId")
    .join(movies, "movieId")
    .select("user_idx", "movie_idx", "label")
)

In [ ]:
num_pos = positive_idx.count()
num_users = users.count()
num_movies = movies.count()

negative_fast = (
    positive_idx
    .select("user_idx")
    .withColumn("movie_idx", floor(rand(seed=42) * num_movies).cast("int"))
    .withColumn("label", lit(0))
)

ncf_data = positive_idx.unionByName(negative_fast).orderBy(rand(seed=42))
ncf_train, ncf_test = ncf_data.randomSplit([0.8, 0.2], seed=42)

{"num_pos": num_pos, "num_users": num_users, "num_movies": num_movies}

In [ ]:
GENOME_FEATURE_DIM = 128

selected_genome_tags = (
    genome_scores
    .join(movies.select("movieId"), "movieId", "inner")
    .groupBy("tagId")
    .agg(stddev("relevance").alias("std_relevance"))
    .orderBy(desc("std_relevance"))
    .limit(GENOME_FEATURE_DIM)
    .join(genome_tags, "tagId", "left")
    .orderBy("tagId")
)

selected_tags_pd = selected_genome_tags.toPandas().reset_index(drop=True)
selected_tags_pd["feature_idx"] = np.arange(len(selected_tags_pd), dtype=np.int32)
selected_tags_pd.to_csv("/workspace/outputs/ncf/genome_feature_tags.csv", index=False)

selected_tag_ids = [int(x) for x in selected_tags_pd["tagId"].tolist()]
tag_id_to_feature_idx = {int(row.tagId): int(row.feature_idx) for row in selected_tags_pd.itertuples(index=False)}

selected_tags_pd.head()

In [ ]:
movies_pd = movies.orderBy("movie_idx").toPandas()
movie_id_to_idx = dict(zip(movies_pd["movieId"].astype(int), movies_pd["movie_idx"].astype(int)))

genome_subset = (
    genome_scores
    .join(movies.select("movieId", "movie_idx"), "movieId", "inner")
    .filter(col("tagId").isin(selected_tag_ids))
    .select("movie_idx", "tagId", "relevance")
)

genome_pd = genome_subset.toPandas()

movie_genome_features = np.zeros((num_movies, len(selected_tag_ids)), dtype=np.float32)
for row in genome_pd.itertuples(index=False):
    feature_idx = tag_id_to_feature_idx[int(row.tagId)]
    movie_genome_features[int(row.movie_idx), feature_idx] = float(row.relevance)

np.save("/workspace/outputs/ncf/movie_genome_features.npy", movie_genome_features)

movie_genome_summary = {
    "num_movies": int(num_movies),
    "genome_feature_dim": int(movie_genome_features.shape[1]),
    "matrix_shape": [int(movie_genome_features.shape[0]), int(movie_genome_features.shape[1])],
    "feature_file": "/workspace/outputs/ncf/movie_genome_features.npy",
    "feature_tags_file": "/workspace/outputs/ncf/genome_feature_tags.csv",
    "feature_selection_strategy": "Top genome tags by relevance standard deviation over movies used in NCF dataset",
}

movie_genome_summary

In [ ]:
ncf_train.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/train_csv"
)

ncf_test.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/test_csv"
)

users.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/user_mapping_csv"
)

movies.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/movie_mapping_csv"
)

In [ ]:
metadata = {
    "num_users": int(num_users),
    "num_movies": int(num_movies),
    "num_positive_samples": int(num_pos),
    "negative_sampling_ratio": 1,
    "label_rule": "rating >= 4.0 => positive",
    "model_input": "user_idx, movie_idx, label, movie_genome_features[movie_idx]",
    "uses_genome_features": True,
    **movie_genome_summary,
}

with open("/workspace/outputs/ncf/ncf_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata

In [ ]:
spark.stop()